# Frozen Qwen3 Embedding + SVM diagnostic

This experiment uses the official Kaggle **Qwen3-Embedding-4B** model as a frozen text representation, then trains a lightweight LinearSVC classifier.

The evaluation is prompt-grouped when `prompt_id` is available, preventing the same prompt from appearing in both train and validation folds. No labels or classifier are used when creating embeddings.


In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.svm import LinearSVC
from transformers import AutoTokenizer, AutoModel

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
TRAIN_PATH = DATA_DIR / 'train.csv'
RESULT_PATH = Path('/kaggle/working/qwen3_svm_results.json')
EMBED_PATH = Path('/kaggle/working/qwen3_embeddings.npz')
SEED = 42
N_SPLITS = 5
BATCH_SIZE = 8
MAX_LENGTH = 512
C_VALUES = [0.1, 0.3, 1.0, 3.0, 10.0]

assert torch.cuda.is_available(), 'GPU is required for the Qwen3 embedding experiment'
train = pd.read_csv(TRAIN_PATH)
assert {'text', 'model'}.issubset(train.columns)
text = train['text'].fillna('').astype(str).tolist()
y = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].astype(str).to_numpy() if 'prompt_id' in train.columns else None
print(f'rows={len(train):,}; classes={len(np.unique(y))}; grouped={groups is not None}; device={torch.cuda.get_device_name(0)}')

# Locate the attached Kaggle model without relying on internet access.
candidates = [
    Path('/kaggle/input/qwen-3-embedding/transformers/4b/1'),
    Path('/kaggle/input/qwen-3-embedding/Transformers/4b/1'),
]
model_dir = next((p for p in candidates if p.exists()), None)
if model_dir is None:
    matches = list(Path('/kaggle/input').glob('**/config.json'))
    qwen_matches = [p.parent for p in matches if 'qwen' in str(p).lower() and 'embedding' in str(p).lower()]
    model_dir = qwen_matches[0] if qwen_matches else None
assert model_dir is not None and model_dir.exists(), 'Attached Qwen3-Embedding model was not found'
print('model_dir=', model_dir)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_dir, padding_side='left', local_files_only=True)
encoder = AutoModel.from_pretrained(
    model_dir,
    torch_dtype=torch.float16,
    local_files_only=True,
).cuda().eval()

def last_token_pool(last_hidden_states, attention_mask):
    left_padding = bool((attention_mask[:, -1].sum() == attention_mask.shape[0]).item())
    if left_padding:
        return last_hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_index = torch.arange(last_hidden_states.shape[0], device=last_hidden_states.device)
    return last_hidden_states[batch_index, sequence_lengths]

@torch.inference_mode()
def encode_texts(items):
    chunks=[]
    for start in range(0, len(items), BATCH_SIZE):
        batch = items[start:start+BATCH_SIZE]
        tokens = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
        tokens = {k:v.cuda() for k,v in tokens.items()}
        outputs = encoder(**tokens)
        pooled = last_token_pool(outputs.last_hidden_state, tokens['attention_mask'])
        pooled = F.normalize(pooled.float(), p=2, dim=1).cpu().numpy()
        chunks.append(pooled)
        if start == 0 or start + BATCH_SIZE >= len(items):
            print(f'encoded {min(start+BATCH_SIZE,len(items)):,}/{len(items):,}')
    return np.vstack(chunks).astype(np.float32)

embeddings = encode_texts(text)
np.savez_compressed(EMBED_PATH, embeddings=embeddings)
print('embedding_shape=', embeddings.shape, 'saved=', EMBED_PATH)
del encoder, tokenizer
torch.cuda.empty_cache(); gc.collect()


In [ ]:
if groups is not None:
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    splits = list(splitter.split(embeddings, y, groups=groups))
else:
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    splits = list(splitter.split(embeddings, y))

all_scores = {}
for C in C_VALUES:
    fold_scores=[]
    for fold, (tr, va) in enumerate(splits, 1):
        clf = LinearSVC(C=C, class_weight='balanced', max_iter=5000, random_state=SEED)
        clf.fit(embeddings[tr], y[tr])
        pred = clf.predict(embeddings[va])
        score = balanced_accuracy_score(y[va], pred)
        fold_scores.append(float(score))
        print(f'C={C:<4} fold={fold} balanced_accuracy={score:.5f}')
    all_scores[str(C)] = {'fold_scores': fold_scores, 'mean': float(np.mean(fold_scores)), 'std': float(np.std(fold_scores))}

best_C = max(all_scores, key=lambda c: all_scores[c]['mean'])
result = {
    'experiment': 'frozen_qwen3_embedding_linear_svc',
    'model_dir': str(model_dir),
    'embedding_shape': list(embeddings.shape),
    'grouped_validation': groups is not None,
    'n_splits': N_SPLITS,
    'scores': all_scores,
    'best_C': float(best_C),
    'best_mean': all_scores[best_C]['mean'],
    'best_std': all_scores[best_C]['std'],
}
RESULT_PATH.write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))


The artifact `qwen3_svm_results.json` contains fold-wise balanced accuracy and the best C value. This experiment is diagnostic only; it does not overwrite the production submission notebook.
